<h1 align="left">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_white.svg" width="500">
    <source media="(prefers-color-scheme: light)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
    <img alt="Logo" src="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
  </picture>
  <h2 align="left">123D: Visualization Tutorial</h1>
</h1>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from py123d.api import SceneFilter, get_filtered_scenes

## 3.1 Download Demo Logs

You can download demo logs for 123D as described in the [documentation](https://autonomousvision.github.io/py123d/installation/). After the installation and download, you can start with the tutorial.


## 3.2 Create Scenes by filtering the datasets

As in other tutorials, we first query some scenes fro visualization. 

In [ ]:
scene_filter = SceneFilter(
    datasets=["physical-ai-av"],
    split_names=None,
    target_iteration_duration_s=0.5,
    future_duration_s=0.3,  # No duration means that the scene will include the complete log.
    history_duration_s=0.5,
    timestamp_threshold_s=1.0,
    shuffle=True,
    has_map=None,  # Only include scenes/logs with an available map API.
    required_scene_modalities=["ego_state_se3", "box_detections_se3", "lidar:all", "camera:all"],
)
scenes = get_filtered_scenes(scene_filter)

dataset_splits = set(scene.log_metadata.split for scene in scenes)
print(f"Found {len(scenes)} scenes from {len(dataset_splits)} datasplits:")
for split in dataset_splits:
    print(f" - {split}")

In [ ]:
from py123d.datatypes import LidarID

scene = scenes[0]
print(scene.dataset)

include_history = True

sync_timestamps = scene.get_all_iteration_timestamps(include_history)
initial_time = scene.get_timestamp_at_iteration(0).time_us

sync_timestamps_ms = (np.array([timestamp.time_us for timestamp in sync_timestamps]) - initial_time) / 1e3

pcam_timestamps = {}
for pinhole_camera_id in scene.available_camera_ids:
    pcam_timestamps[pinhole_camera_id] = [
        timestamp.time_us
        for timestamp in scene.get_all_camera_timestamps(pinhole_camera_id, include_history=include_history)
    ]
    pcam_timestamps[pinhole_camera_id] = np.array(pcam_timestamps[pinhole_camera_id], dtype=np.int64)
    pcam_timestamps[pinhole_camera_id] = (pcam_timestamps[pinhole_camera_id] - initial_time).astype(np.float64) / 1e3

lidar_start_timestamp = [
    timestamp.time_us
    for timestamp in scene.get_all_lidar_timestamps(LidarID.LIDAR_MERGED, include_history=include_history)
]
lidar_start_timestamp = (np.array(lidar_start_timestamp) - initial_time) / 1e3

ego_timestamps = [
    timestamp.time_us for timestamp in scene.get_all_ego_state_se3_timestamps(include_history=include_history)
]
ego_timestamps = (np.array(ego_timestamps) - initial_time) / 1e3

box_timestamps = [
    timestamp.time_us for timestamp in scene.get_all_box_detections_se3_timestamps(include_history=include_history)
]
box_timestamps = (np.array(box_timestamps) - initial_time) / 1e3

# Sort cameras by earliest timestamp
pcam_timestamps = dict(sorted(pcam_timestamps.items(), key=lambda item: item[1][0]))

fig, ax = plt.subplots(figsize=(12, 6))
half_height = 0.35

# Explicit colors per modality
ego_color = "tab:blue"
box_color = "tab:orange"
lidar_color = "tab:green"


# Add sync timestamps as scattered thin reference lines
ax.vlines(
    sync_timestamps_ms,
    ymin=-1,
    ymax=len(pcam_timestamps) + 2.5,
    color="gray",
    alpha=1,
    linewidth=0.5,
    linestyle="--",
    label="Sync Timestamps",
)

alpha = 1.0
linewidth = 2

yticklabels = []
y_offset = 0

if len(ego_timestamps) > 0:
    ax.vlines(
        ego_timestamps,
        ymin=y_offset - half_height,
        ymax=y_offset + half_height,
        label="Ego State",
        color=ego_color,
        alpha=alpha,
        linewidth=linewidth,
    )
    y_offset += 1
    yticklabels.append("Ego State")

if len(box_timestamps) > 0:
    ax.vlines(
        box_timestamps,
        ymin=y_offset - half_height,
        ymax=y_offset + half_height,
        label="Box Detections",
        color=box_color,
        alpha=alpha,
        linewidth=linewidth,
    )
    y_offset += 1
    yticklabels.append("Box Detections")


if len(lidar_start_timestamp) > 0:
    ax.vlines(
        lidar_start_timestamp,
        ymin=y_offset - half_height,
        ymax=y_offset + half_height,
        label="Lidar Start",
        color="tab:green",
        alpha=alpha,
        linewidth=linewidth,
    )
    y_offset += 1
    yticklabels.append("Lidar Start")


for i, (camera_id, timestamps) in enumerate(pcam_timestamps.items()):
    y = y_offset + i

    ax.vlines(
        timestamps,
        ymin=y - half_height,
        ymax=y + half_height,
        label=camera_id.name,
        color=f"C{int(camera_id) + 3 % 20}",
        alpha=alpha,
        linewidth=linewidth,
    )
    yticklabels.append(camera_id.name)


ax.set_xlabel("Timestamp (ms)")
ax.set_yticks(range(len(yticklabels)))
ax.set_yticklabels(yticklabels)
ax.set_title("Sensor Timestamps")
ax.legend(loc="upper right")
ax.set_xlim(-610, 520)

plt.tight_layout()
plt.show()

In [ ]:
import time

from py123d.api.scene.arrow.arrow_scene_api import ArrowSceneAPI

assert isinstance(scene, ArrowSceneAPI)

start = time.time()
result = scene.get_modality_column_at_iteration(
    iteration=0,
    column="timestamp_us",
    modality_type="box_detections_se3",
    modality_id=None,
    deserialize=True,
)
end = time.time()
print(f"Time taken: {(end - start) * 1e3:.2f} ms")

result

In [ ]:
start = time.time()
result = scene.get_lidar_at_iteration(
    iteration=0,
    lidar_id=LidarID.LIDAR_MERGED,
)
end = time.time()
print(f"Time taken: {(end - start) * 1e3:.2f} ms")

result